# 🔍 Exploración de Capas GeoPackage
Notebook para inspeccionar estructura, columnas y estadísticas de archivos `.gpkg`

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import fiona

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Listar capas disponibles en un archivo

In [ ]:
# Cambia la ruta al archivo que quieras explorar
ARCHIVO = 'MUNICIPAL_CARACTERISTICAS.gpkg'

capas = fiona.listlayers(ARCHIVO)
print(f'Capas en {ARCHIVO}:')
for i, c in enumerate(capas):
    print(f'  [{i}] {c}')

## 2. Cargar y ver estructura básica

In [ ]:
# Si el archivo tiene varias capas, especifica layer='nombre'
gdf = gpd.read_file(ARCHIVO)

print(f'Filas:    {len(gdf):,}')
print(f'Columnas: {len(gdf.columns)}')
print(f'CRS:      {gdf.crs}')
print(f'Tipo geo: {gdf.geometry.geom_type.unique()}')
print()
print('Columnas:')
for col in gdf.columns:
    print(f'  {col:<35} {str(gdf[col].dtype):<12} nulls={gdf[col].isna().sum()}')

## 3. Primeras filas

In [ ]:
gdf.drop(columns='geometry').head(10)

## 4. Estadísticas de columnas numéricas

In [ ]:
gdf.drop(columns='geometry').describe().T

## 5. Valores únicos de columnas categóricas

In [ ]:
# Columnas con pocos valores únicos (candidatas a ser categorías)
for col in gdf.drop(columns='geometry').columns:
    n = gdf[col].nunique()
    if 2 <= n <= 20:
        print(f'\n{col} ({n} valores):')
        print(gdf[col].value_counts().to_string())

## 6. Explorar archivo de AGEBs (nuevo)

In [ ]:
# Cambia el nombre al archivo que te mandó Pili
ARCHIVO_AGEBS = 'NOMBRE_DEL_ARCHIVO.gpkg'  # <-- cambia aquí

gdf_ageb = gpd.read_file(ARCHIVO_AGEBS)
print(f'Filas: {len(gdf_ageb):,} | CRS: {gdf_ageb.crs}')
gdf_ageb.drop(columns='geometry').head(5)

## 7. Análisis de columnas de tiempo y categorías (AGEBs)

In [ ]:
# Columnas de tiempo en minutos
cols_tiempo = ['min_pna', 'min_camas', 'min_sna']
cols_cat    = ['cat_PNA', 'cat_camas', 'cat_sna']

print('=== Estadísticas de tiempo (minutos) ===')
display(gdf_ageb[cols_tiempo].describe())

print('\n=== Distribución por categoría ===')
for col in cols_cat:
    if col in gdf_ageb.columns:
        print(f'\n{col}:')
        print(gdf_ageb[col].value_counts().to_string())

## 8. Población por categoría de acceso

In [ ]:
# Columnas de población — ajusta el nombre si es diferente
col_pob = 'POBLACION TOTAL'  # o 'p_hl3ms', 'POB_TOTAL', etc.

for raster, col_cat in [('PNA', 'cat_PNA'), ('Camas (H. no espec.)', 'cat_camas'), ('SNA (todos hosp.)', 'cat_sna')]:
    if col_cat in gdf_ageb.columns and col_pob in gdf_ageb.columns:
        print(f'\n📊 Población por categoría — {raster}')
        tabla = gdf_ageb.groupby(col_cat)[col_pob].agg(['sum','count'])
        tabla.columns = ['Población total', 'AGEBs']
        tabla['% Población'] = tabla['Población total'] / tabla['Población total'].sum() * 100
        tabla = tabla.sort_index()
        display(tabla.style.format({'Población total': '{:,.0f}', 'AGEBs': '{:,}', '% Población': '{:.1f}%'}))

## 9. Buscar columna de población automáticamente

In [ ]:
# Si no sabes el nombre exacto de la columna de población
candidatas = [c for c in gdf_ageb.columns 
              if any(k in c.lower() for k in ['pob','pop','total','hab'])]
print('Columnas candidatas de población:')
for c in candidatas:
    print(f'  {c}: suma={gdf_ageb[c].sum():,.0f}, max={gdf_ageb[c].max():,.0f}')